# CFD AirfRANS Training Notebook

这份 notebook 单独复用了仓库里已经存在的训练、评测、推理脚本，不修改本地 Python 源码。

适用目标：

- 在本地或 Google Colab 上完成 AirfRANS 版 CFD 代理模型训练
- 自动根据运行环境选择 `GPU` 或 `CPU`
- 重新做 `70 / 15 / 15` 的训练、验证、测试集切分
- 调用现有 `train.py / evaluate.py / infer.py` 生成结果
- 在 notebook 末尾直接显示关键图像与指标

本 notebook 的默认配置与当前项目主线保持一致：

- `1024` 个 query points
- `256` 个 surface points
- `latent_dim = 256`
- 开启 `physics loss`
- 训练 `30` 轮

如果你在 Colab 中运行，请先确保整个仓库已经位于运行环境中，例如挂载 Google Drive 后进入仓库目录，或者直接把仓库同步到 `/content/CFD`。

## 1. 环境准备与依赖说明

这一部分的功能：

- 自动识别是否在 Google Colab 中运行
- 自动查找仓库根目录
- 检查基础依赖是否齐全
- 以 `editable` 方式安装当前项目，保证 notebook 可以直接调用 `src/` 下的包

参数说明：

- `REPO_ROOT_HINT`：如果自动定位不到仓库，可以手动指定仓库根目录
- `INSTALL_EDITABLE`：是否执行 `pip install -e . --no-deps`
- `INSTALL_MISSING_PACKAGES`：是否自动安装缺失的基础依赖

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPO_ROOT_HINT = None
# Colab 如果自动找不到仓库，就把这里改成你的实际路径，例如：
# REPO_ROOT_HINT = '/content/drive/MyDrive/CFD'
INSTALL_EDITABLE = True
INSTALL_MISSING_PACKAGES = True
MOUNT_DRIVE_IN_COLAB = True

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB and MOUNT_DRIVE_IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

def is_repo_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src').exists() and (path / 'scripts').exists()

def find_repo_root(repo_root_hint=None):
    candidates = []
    if repo_root_hint:
        candidates.append(Path(repo_root_hint).expanduser())

    common_roots = [
        Path.cwd(),
        *Path.cwd().parents,
        Path('/content/CFD'),
        Path('/content/cfd'),
        Path('/content/project'),
        Path('/content/drive/MyDrive/CFD'),
        Path('/content/drive/MyDrive/cfd'),
        Path('/content/drive/MyDrive/project/CFD'),
        Path('/content/drive/MyDrive'),
        Path('/content'),
    ]
    candidates.extend(common_roots)

    seen = set()
    checked = []

    for candidate in candidates:
        candidate = candidate.resolve() if candidate.exists() else candidate
        if str(candidate) in seen:
            continue
        seen.add(str(candidate))
        checked.append(str(candidate))

        if candidate.exists() and is_repo_root(candidate):
            return candidate.resolve(), checked

        if candidate.exists() and candidate.is_dir():
            for subdir in candidate.iterdir():
                if not subdir.is_dir():
                    continue
                checked.append(str(subdir))
                if is_repo_root(subdir):
                    return subdir.resolve(), checked

    return None, checked

REPO_ROOT, CHECKED_PATHS = find_repo_root(REPO_ROOT_HINT)
if REPO_ROOT is None:
    print('已检查以下目录：')
    for item in CHECKED_PATHS[:50]:
        print(' -', item)
    raise FileNotFoundError(
        '未找到仓库根目录。\n'
        '如果你在 Colab 里把仓库放在 Google Drive，请先确认 Drive 已挂载，\n'
        '然后把 REPO_ROOT_HINT 改成你的实际仓库路径，例如：\n'
        "REPO_ROOT_HINT = '/content/drive/MyDrive/CFD'"
    )

os.chdir(REPO_ROOT)
print(f'IS_COLAB={IS_COLAB}')
print(f'REPO_ROOT={REPO_ROOT}')

required_packages = {
    'numpy': 'numpy>=1.26',
    'scipy': 'scipy>=1.12',
    'pandas': 'pandas>=2.1',
    'matplotlib': 'matplotlib>=3.8',
    'sklearn': 'scikit-learn>=1.4',
    'yaml': 'pyyaml>=6.0',
    'pydantic': 'pydantic>=2.6',
    'torch': 'torch>=2.1',
    'h5py': 'h5py>=3.10',
}

missing = [package for module_name, package in required_packages.items() if importlib.util.find_spec(module_name) is None]
if missing and INSTALL_MISSING_PACKAGES:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
elif missing:
    print('Missing packages detected but not installed:', missing)

if INSTALL_EDITABLE:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    print('Installed current project in editable mode.')


## 2. 训练参数配置

这一部分定义本轮训练和评测的主要参数。

功能说明：

- 设置实验名称和输出目录
- 指定基础数据集与重切分数据集路径
- 定义训练轮数、batch size、网络宽度和 loss 开关
- 自动识别当前是 GPU 还是 CPU 运行

关键参数说明：

- `EXPERIMENT_NAME`：本次运行在 `outputs/` 下的输出目录名
- `SOURCE_DATASET`：原始 heavy 数据集
- `RESPLIT_DATASET`：新的 `70/15/15` 切分数据文件
- `EPOCHS`：训练轮数
- `NUM_QUERY_POINTS`：每个样本的体场查询点数
- `NUM_SURFACE_POINTS`：每个样本的表面点数
- `LATENT_DIM`：DeepONet 潜空间维度
- `USE_PHYSICS`：是否启用 physics loss
- `PHYSICS_WEIGHT`：physics loss 权重
- `USE_SLICE_LOSS` / `USE_FEATURE_LOSS`：是否启用 slice 和 feature 监督

In [ ]:
import json
import torch

EXPERIMENT_NAME = 'airfrans_notebook_run'
CONFIG_PATH = REPO_ROOT / 'configs' / 'experiments' / 'airfrans_original_heavy.yaml'
SOURCE_DATASET = REPO_ROOT / 'outputs' / 'data' / 'airfrans_original_heavy.npz'
RESPLIT_DATASET = REPO_ROOT / 'outputs' / 'data' / 'airfrans_original_heavy_resplit_colab.npz'
RESPLIT_METADATA = REPO_ROOT / 'outputs' / 'data' / 'airfrans_original_heavy_resplit_colab.json'

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

EPOCHS = 30
EARLY_STOPPING_PATIENCE = 12
BATCH_SIZE = 4
LEARNING_RATE = 1e-3

NUM_QUERY_POINTS = 1024
NUM_SURFACE_POINTS = 256
LATENT_DIM = 256
HIDDEN_DIM = 256
FEATURE_OUTPUT_DIM = 2

USE_PHYSICS = True
PHYSICS_WEIGHT = 0.02
USE_SLICE_LOSS = True
SLICE_WEIGHT = 0.05
USE_FEATURE_LOSS = True
FEATURE_WEIGHT = 0.02

NUM_VIS_SAMPLES = 3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if DEVICE == 'cuda':
    print('Using GPU:', torch.cuda.get_device_name(0))
else:
    print('Using CPU runtime.')

run_summary = {
    'experiment_name': EXPERIMENT_NAME,
    'config_path': str(CONFIG_PATH),
    'source_dataset': str(SOURCE_DATASET),
    'resplit_dataset': str(RESPLIT_DATASET),
    'device': DEVICE,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'num_query_points': NUM_QUERY_POINTS,
    'num_surface_points': NUM_SURFACE_POINTS,
    'latent_dim': LATENT_DIM,
    'hidden_dim': HIDDEN_DIM,
    'use_physics': USE_PHYSICS,
    'physics_weight': PHYSICS_WEIGHT,
    'use_slice_loss': USE_SLICE_LOSS,
    'slice_weight': SLICE_WEIGHT,
    'use_feature_loss': USE_FEATURE_LOSS,
    'feature_weight': FEATURE_WEIGHT,
}
print(json.dumps(run_summary, indent=2, ensure_ascii=False))


## 3. 重新切分 AirfRANS 数据集

这一部分的功能：

- 读取当前仓库已有的 `airfrans_original_heavy.npz`
- 生成新的随机 `70 / 15 / 15` 切分
- 保存为一个新的 `.npz` 文件，不覆盖原始数据文件
- 同时输出一份 `.json` 元数据，记录本次切分口径

说明：

- notebook 不修改原数据，只创建一个新的切分文件
- 如果新的切分文件已经存在，则直接复用，避免重复处理

In [ ]:
import numpy as np

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-8

SOURCE_DATASET.parent.mkdir(parents=True, exist_ok=True)

if not SOURCE_DATASET.exists():
    raise FileNotFoundError(f'未找到基础数据集: {SOURCE_DATASET}')

if not RESPLIT_DATASET.exists():
    raw = np.load(SOURCE_DATASET, allow_pickle=True)
    num_samples = raw['branch_inputs'].shape[0]
    rng = np.random.default_rng(SEED)
    indices = rng.permutation(num_samples)

    n_train = int(num_samples * TRAIN_RATIO)
    n_val = int(num_samples * VAL_RATIO)
    train_indices = np.sort(indices[:n_train].astype(np.int64))
    val_indices = np.sort(indices[n_train:n_train + n_val].astype(np.int64))
    test_indices = np.sort(indices[n_train + n_val:].astype(np.int64))

    payload = {key: raw[key] for key in raw.files if key not in {'train_indices', 'val_indices', 'test_indices', 'test_unseen_geometry_indices', 'test_unseen_condition_indices'}}
    payload['train_indices'] = train_indices
    payload['val_indices'] = val_indices
    payload['test_indices'] = test_indices

    np.savez_compressed(RESPLIT_DATASET, **payload)

    metadata = {
        'dataset_path': str(RESPLIT_DATASET.relative_to(REPO_ROOT)),
        'source': 'airfrans_original_resplit_notebook',
        'based_on': str(SOURCE_DATASET.relative_to(REPO_ROOT)),
        'num_samples': int(num_samples),
        'num_query_points': int(raw['query_points'].shape[1]),
        'num_surface_points': int(raw['surface_points'].shape[1]),
        'train_size': int(train_indices.shape[0]),
        'val_size': int(val_indices.shape[0]),
        'test_size': int(test_indices.shape[0]),
        'split_strategy': f'random_global_70_15_15_seed_{SEED}',
    }
    RESPLIT_METADATA.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Created resplit dataset.')
else:
    print('Reuse existing resplit dataset.')

print(RESPLIT_DATASET)
print(RESPLIT_METADATA)
print(RESPLIT_METADATA.read_text(encoding='utf-8'))


## 4. 启动训练

这一部分直接调用现有的 [scripts/train.py](/Users/jason/Documents/CFD/scripts/train.py) 训练入口。

功能说明：

- 不修改项目源码，只通过 `--override` 传参
- 自动把 `experiment.device` 设为当前运行环境检测到的 `cuda` 或 `cpu`
- 使用新的重切分数据集进行训练
- 开启 `physics loss`、`slice loss` 和 `feature loss`

输出结果会保存在：

- `outputs/<EXPERIMENT_NAME>/checkpoints/`
- `outputs/<EXPERIMENT_NAME>/reports/`

如果你只想先快速试跑，可以把前面的 `EPOCHS` 改小，例如 `5` 或 `10`。

In [ ]:
def run_command(command, cwd=REPO_ROOT):
    print('Running command:')
    print(' '.join(command))
    subprocess.run(command, cwd=cwd, check=True)

train_overrides = [
    f'experiment.name={EXPERIMENT_NAME}',
    f'experiment.device={DEVICE}',
    'data.dataset_type=file',
    f'data.dataset_path={RESPLIT_DATASET}',
    'data.strict_quality_checks=false',
    f'data.num_query_points={NUM_QUERY_POINTS}',
    f'data.num_surface_points={NUM_SURFACE_POINTS}',
    f'model.hidden_dim={HIDDEN_DIM}',
    f'model.latent_dim={LATENT_DIM}',
    f'model.feature_output_dim={FEATURE_OUTPUT_DIM}',
    f'train.epochs={EPOCHS}',
    f'train.batch_size={BATCH_SIZE}',
    f'train.learning_rate={LEARNING_RATE}',
    f'train.early_stopping_patience={EARLY_STOPPING_PATIENCE}',
    f'loss.use_physics={str(USE_PHYSICS).lower()}',
    f'loss.physics_weight={PHYSICS_WEIGHT}',
    f'loss.use_slice_loss={str(USE_SLICE_LOSS).lower()}',
    f'loss.slice_weight={SLICE_WEIGHT}',
    f'loss.use_feature_loss={str(USE_FEATURE_LOSS).lower()}',
    f'loss.feature_weight={FEATURE_WEIGHT}',
]

train_command = [sys.executable, 'scripts/train.py', '--config', str(CONFIG_PATH)]
for item in train_overrides:
    train_command.extend(['--override', item])

run_command(train_command)

RUN_DIR = REPO_ROOT / 'outputs' / EXPERIMENT_NAME
BEST_CHECKPOINT = RUN_DIR / 'checkpoints' / 'best.pt'
HISTORY_CSV = RUN_DIR / 'reports' / 'history.csv'
print('RUN_DIR =', RUN_DIR)
print('BEST_CHECKPOINT =', BEST_CHECKPOINT)
print('HISTORY_CSV =', HISTORY_CSV)


## 5. 评测与单样本推理

这一部分会连续调用：

- `scripts/evaluate.py`：在测试集上统计指标、导出图像和分析文件
- `scripts/infer.py`：对示例输入做单样本推理，并导出 `analysis bundle`

功能说明：

- 评测会在测试集上输出 `field / scalar / surface / slice / feature` 指标
- 推理会导出 `predictions.json`、`surface_values.csv`、`slice_values.csv`、`feature_summary.json`
- 所有可视化图像都保存在输出目录中，后面会直接在 notebook 中显示

In [ ]:
EVAL_DIR = REPO_ROOT / 'outputs' / EXPERIMENT_NAME / 'eval' / 'test'
INFER_OUTPUT = REPO_ROOT / 'outputs' / EXPERIMENT_NAME / 'inference.json'
ANALYSIS_BUNDLE_DIR = REPO_ROOT / 'outputs' / EXPERIMENT_NAME / 'analysis_bundle'
INFERENCE_INPUT = REPO_ROOT / 'examples' / 'inference_input.json'

eval_overrides = [
    f'experiment.name={EXPERIMENT_NAME}',
    f'experiment.device={DEVICE}',
    'data.dataset_type=file',
    f'data.dataset_path={RESPLIT_DATASET}',
    'data.strict_quality_checks=false',
    f'model.feature_output_dim={FEATURE_OUTPUT_DIM}',
    f'loss.use_physics={str(USE_PHYSICS).lower()}',
    f'loss.physics_weight={PHYSICS_WEIGHT}',
    f'loss.use_slice_loss={str(USE_SLICE_LOSS).lower()}',
    f'loss.slice_weight={SLICE_WEIGHT}',
    f'loss.use_feature_loss={str(USE_FEATURE_LOSS).lower()}',
    f'loss.feature_weight={FEATURE_WEIGHT}',
    'eval.split_name=test',
    'eval.save_plots=true',
    'eval.export_analysis=true',
    f'eval.num_visualization_samples={NUM_VIS_SAMPLES}',
]

evaluate_command = [
    sys.executable,
    'scripts/evaluate.py',
    '--config',
    str(CONFIG_PATH),
    '--checkpoint',
    str(BEST_CHECKPOINT),
]
for item in eval_overrides:
    evaluate_command.extend(['--override', item])
run_command(evaluate_command)

infer_command = [
    sys.executable,
    'scripts/infer.py',
    '--checkpoint',
    str(BEST_CHECKPOINT),
    '--input',
    str(INFERENCE_INPUT),
    '--output',
    str(INFER_OUTPUT),
    '--export-dir',
    str(ANALYSIS_BUNDLE_DIR),
]
run_command(infer_command)

print('EVAL_DIR =', EVAL_DIR)
print('INFER_OUTPUT =', INFER_OUTPUT)
print('ANALYSIS_BUNDLE_DIR =', ANALYSIS_BUNDLE_DIR)


## 6. 结果读取与指标表格

这一部分从评测输出目录中读取：

- `metrics.json`
- `history.csv`
- `analysis_bundle` 中的导出文件

功能说明：

- 用表格形式展示核心测试指标
- 读取训练曲线历史并找到最佳验证轮次
- 抽取表面压力系数等分析信息，便于快速检查模型输出是否正常

In [ ]:
import csv
import pandas as pd
from IPython.display import display

metrics_path = EVAL_DIR / 'metrics.json'
report_path = EVAL_DIR / 'report.md'
surface_values_path = ANALYSIS_BUNDLE_DIR / 'surface_values.csv'
feature_summary_path = ANALYSIS_BUNDLE_DIR / 'feature_summary.json'

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
metrics_frame = pd.DataFrame([
    {'metric': key, 'value': value} for key, value in metrics.items()
]).sort_values('metric').reset_index(drop=True)
display(metrics_frame)

history_frame = pd.read_csv(HISTORY_CSV)
best_row = history_frame.loc[history_frame['val_loss_total'].idxmin()]
display(pd.DataFrame([best_row]))

surface_frame = pd.read_csv(surface_values_path)
surface_summary = pd.DataFrame([
    {
        'column': column,
        'min': surface_frame[column].min(),
        'max': surface_frame[column].max(),
        'mean': surface_frame[column].mean(),
    }
    for column in surface_frame.columns
    if column != 'point_index'
])
display(surface_summary)

feature_summary = json.loads(feature_summary_path.read_text(encoding='utf-8'))
print(json.dumps(feature_summary.get('high_gradient_region_summary', {}), indent=2, ensure_ascii=False))


## 7. 可视化结果展示

这一部分直接从评测目录读取已经导出的 PNG 图片，并在 notebook 中展示。

功能说明：

- 显示训练损失曲线
- 显示 `Cl / Cd` 散点图
- 显示测试样本的 `surface Cp`、`slice p` 和高梯度区域图
- 所有图像都来自现有 Python 脚本实际导出的结果，不是 notebook 临时重画的占位图

In [ ]:
from IPython.display import Image, Markdown

image_paths = [
    EVAL_DIR / 'loss_curve.png',
    EVAL_DIR / 'cl_scatter.png',
    EVAL_DIR / 'cd_scatter.png',
    EVAL_DIR / 'sample_00' / 'surface_cp.png',
    EVAL_DIR / 'sample_00' / 'surface_pressure.png',
    EVAL_DIR / 'sample_00' / 'slice_p.png',
    EVAL_DIR / 'sample_00' / 'high_gradient_regions.png',
]

for path in image_paths:
    display(Markdown(f'### {path.name}'))
    display(Image(filename=str(path)))


## 8. 结果文件位置

运行完成后，重点结果文件通常位于：

- `outputs/<EXPERIMENT_NAME>/checkpoints/best.pt`
- `outputs/<EXPERIMENT_NAME>/reports/history.csv`
- `outputs/<EXPERIMENT_NAME>/eval/test/metrics.json`
- `outputs/<EXPERIMENT_NAME>/eval/test/report.md`
- `outputs/<EXPERIMENT_NAME>/analysis_bundle/`

如果你后续还要继续做实验，可以直接复制这个 notebook，修改参数区的变量，然后再次运行。

In [ ]:
artifacts = [
    BEST_CHECKPOINT,
    HISTORY_CSV,
    EVAL_DIR / 'metrics.json',
    EVAL_DIR / 'report.md',
    INFER_OUTPUT,
    ANALYSIS_BUNDLE_DIR,
]

for artifact in artifacts:
    print(artifact)
